# Causal Inference Toolkit — Demo: Lalonde (NSW/PSID)

## Lalonde's Job Training Program Evaluation

Classic observational study estimating the effect of a job training program (NSW) on earnings.
Original data: 185 treated, 15,992 controls (PSID).

**Key challenge**: Severe selection bias — treated group is very different from controls.

**Ground truth**: RCT subset available for comparison (LaLonde 1986).

In [ ]:
import sys
sys.path.insert(0, '../../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from causal_toolkit import (
    CausalModel,
    DoWhyWrapper,
    EconMLWrapper,
    SensitivityAnalyzer,
    ABTestAnalyzer,
    UpliftModeler,
    ForestPlot,
    LovePlot,
    evaluate_uplift,
)
from causal_toolkit.core.base import EstimatorType, IdentificationStrategy, RefutationMethod
from causal_toolkit.utils.data import load_dataset, create_synthetic_data

np.random.seed(42)
sns.set_style('whitegrid')

In [ ]:
# Load Lalonde dataset
df = load_dataset('lalonde')
print(f'Data shape: {df.shape}')
print(f'Treatment balance: {df["treatment"].mean():.2%}')
print(f'Outcome (re78) mean: {df["outcome"].mean():.0f}')
print(f'Treated outcome: {df[df["treatment"]==1]["outcome"].mean():.0f}')
print(f'Control outcome: {df[df["treatment"]==0]["outcome"].mean():.0f}')
df.head()

In [ ]:
# Variables
treatment = 'treatment'
outcome = 'outcome'
confounders = ['age', 'education', 'black', 'hispanic', 'married', 'nodegree', 're74', 're75']
print(f'Confounders: {confounders}')

## 1. Naive Comparison (Biased)

In [ ]:
# Raw difference in means
naive = df[df[treatment]==1][outcome].mean() - df[df[treatment]==0][outcome].mean()
print(f'Naive difference: {naive:.0f}')

# True RCT effect was ~$1,500-2,000
print(f'Expected RCT effect: ~$1,500-2,000')

## 2. Propensity Score Methods with DoWhy

In [ ]:
# Build model
model = CausalModel(
    data=df,
    treatment=treatment,
    outcome=outcome,
    common_causes=confounders,
)
dowhy = DoWhyWrapper(model)

# Identify
estimand = dowhy.identify(IdentificationStrategy.BACKDOOR)
print(f'Adjustment set: {estimand.adjustment_set}')

In [ ]:
# Compare multiple estimators
estimators = [
    EstimatorType.LINEAR_REGRESSION,
    EstimatorType.PROPENSITY_SCORE_MATCHING,
    EstimatorType.PROPENSITY_SCORE_WEIGHTING,
    EstimatorType.PROPENSITY_SCORE_STRATIFICATION,
    EstimatorType.DOUBLY_ROBUST,
]

results = {}
for est_type in estimators:
    try:
        est = dowhy.estimate(est_type)
        results[est_type.value] = est
        print(f'{est_type.value:>30s}: {est.value:>8.0f} [{est.ci_lower:.0f}, {est.ci_upper:.0f}]')
    except Exception as e:
        print(f'{est_type.value:>30s}: ERROR - {e}')

## 3. Sensitivity Analysis — How strong must confounding be?

In [ ]:
# Pick best estimate (e.g., doubly robust)
best_est = results.get('doubly_robust')
if best_est is None:
    best_est = results.get('linear_regression')

# Full sensitivity suite
analyzer = SensitivityAnalyzer(model)
analyzer.cinelli_hazlett(best_est, benchmark_covariates='age')
analyzer.rosenbaum_bounds(best_est)
analyzer.e_value(best_est)

print(analyzer.summarize())

## 4. Covariate Balance Diagnostics

In [ ]:
# SMDs before adjustment
from causal_toolkit.utils.preprocessing import compute_all_smds

smds_before = compute_all_smds(df, treatment, confounders)

# After propensity weighting
from causal_toolkit.utils.preprocessing import propensity_score, inverse_probability_weighting

ps = propensity_score(df, treatment, confounders, model='logistic')
weights = inverse_probability_weighting(df[treatment].values, ps, stabilized=True)

# Weighted SMDs
def weighted_smd(treated_vals, control_vals, weights):
    wt = weights[df[treatment]==1]
    wc = weights[df[treatment]==0]
    if len(wt) == 0 or len(wc) == 0:
        return np.nan
    mean_t = np.average(treated_vals, weights=wt)
    mean_c = np.average(control_vals, weights=wc)
    var_t = np.average((treated_vals - mean_t)**2, weights=wt)
    var_c = np.average((control_vals - mean_c)**2, weights=wc)
    pooled = np.sqrt((var_t + var_c) / 2)
    return (mean_t - mean_c) / pooled if pooled > 0 else np.nan

smds_after = {}
for cov in confounders:
    smd = weighted_smd(df[df[treatment]==1][cov].values,
                       df[df[treatment]==0][cov].values,
                       weights)
    smds_after[cov] = {'smd': smd, 'balanced': abs(smd) < 0.1, 'threshold': 0.1}

# Love plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

love = LovePlot()
love.plot(smds_before, ax=axes[0])
axes[0].set_title('Before Adjustment')
love.plot(smds_after, ax=axes[1])
axes[1].set_title('After IPW Adjustment')

plt.tight_layout()
plt.show()

## 5. Heterogeneous Treatment Effects (CATE)

In [ ]:
# EconML for CATE
econml = EconMLWrapper(
    data=df,
    treatment=treatment,
    outcome=outcome,
    covariates=confounders,
)

# Multiple CATE estimators
cate_estimators = [
    EstimatorType.T_LEARNER,
    EstimatorType.R_LEARNER,
    EstimatorType.CAUSAL_FOREST_CATE,
]

cate_results = {}
for est_type in cate_estimators:
    try:
        est = econml.estimate_cate(est_type)
        cate_results[est_type.value] = est.value
        print(f'{est_type.value:>20s}: mean={np.mean(est.value):.0f}, std={np.std(est.value):.0f}')
    except Exception as e:
        print(f'{est_type.value:>20s}: ERROR - {e}')

## 6. Forest Plot: Compare All Estimates

In [ ]:
# Forest plot
forest = ForestPlot(figsize=(10, 8))

forest_data = []
forest_data.append({
    'label': 'Naive (no adjustment)',
    'estimate': naive,
    'ci_lower': naive - 500,
    'ci_upper': naive + 500,
})

for name, est in results.items():
    forest_data.append({
        'label': name,
        'estimate': est.value,
        'ci_lower': est.ci_lower,
        'ci_upper': est.ci_upper,
    })

# Add RCT benchmark
forest_data.append({
    'label': 'RCT benchmark (LaLonde)',
    'estimate': 1800,
    'ci_lower': 800,
    'ci_upper': 2800,
    'is_overall': True,
})

fig = forest.plot(forest_data, xlabel='Effect on 1978 Earnings ($)',
                  title='LaLonde Job Training: Estimator Comparison')
plt.show()